In [1]:
!pip install -q instructor weave openai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.0/59.

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone --branch main https://github.com/wandb/eval-course
    %cd eval-course

    !pip install -q -r requirements.txt

    %cd notebooks
else:
    print("Not running in Google Colab.")

Cloning into 'eval-course'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 271 (delta 115), reused 96 (delta 86), pack-reused 127 (from 1)
Receiving objects: 100% (271/271), 1.19 MiB | 18.71 MiB/s, done.
Resolving deltas: 100% (167/167), done.
/content/eval-course
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... 

In [3]:
!pwd
!find .. -maxdepth 3 -type f | grep "utils"

/content/eval-course/notebooks
../notebooks/utils/render.py
../notebooks/utils/evals.py
../notebooks/utils/prompts.py
../notebooks/utils/llm_client.py
../notebooks/utils/deserialize.py
../notebooks/utils/config.py
../notebooks/utils/utils.py


In [4]:
from utils.config import ENTITY, WEAVE_PROJECT
from utils.prompts import medical_system_prompt, medical_task

print(ENTITY)
print(WEAVE_PROJECT)

None
eval_course_ch1


In [5]:
from typing import Dict, List, Literal, Optional, Tuple

import instructor
import openai
import pandas as pd
import weave
from pydantic import BaseModel, Field

import os
import getpass

# Prompts you to paste your key securely without saving it in the code
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")
os.environ["WANDB_API_KEY"] = getpass.getpass("Enter your W&B API Key for Weave: ")

Enter your OpenAI API Key: ··········
Enter your W&B API Key for Weave: ··········


In [10]:
from utils.config import ENTITY, WEAVE_PROJECT

ENTITY = "thesisdfki-dfki"
WEAVE_PROJECT = "eval-course"

In [11]:
weave.init(f"{ENTITY}/{WEAVE_PROJECT}")

weave version 0.53.9 is available!  To upgrade, please run:
 $ pip install weave --upgrade
Logged in as Weights & Biases user: thesisdfki.
View Weave data at https://wandb.ai/thesisdfki-dfki/eval-course/weave


In [12]:
N_SAMPLES = 67

In [13]:
from utils.prompts import medical_system_prompt, medical_task

In [20]:
# Initialize the standard OpenAI client
base_client = openai.OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Wrap it with instructor using the modern 1.x syntax
client = instructor.from_openai(base_client)

# Use the high-quota Lite model to avoid hitting the 20 RPD limit
MODEL = "gemini-1.5-flash" # Use the standard public naming format
medical_dataset_url = "https://raw.githubusercontent.com/wyim/aci-bench/main/data/challenge_data/train.csv"

In [21]:
def load_medical_data(url: str, num_samples: int = N_SAMPLES) -> List[Dict]:
    df = pd.read_csv(url)
    print(df.shape)
    samples = df.sample(n=num_samples, random_state=42)
    return samples.to_dict("records")

In [22]:
samples = load_medical_data(medical_dataset_url)

(67, 4)


In [23]:
samples[0]

{'dataset': 'aci',
 'encounter_id': 'D2N037',
 'dialogue': "[doctor] hey dylan what's going on so i lift quite a bit of weights i try to stay in shape as much as i can i'm not like normal people i lift heavy weights and my elbow is extremely sore which elbow is it\n[patient] actually it's both my elbows but my right elbow is hurting me the most\n[doctor] okay and you said you lift a lot of weights\n[patient] mm-hmm\n[doctor] did you play any sports when you were younger\n[patient] no anything you can think of primarily it was basketball baseball and football\n[doctor] okay and did your elbows hurt at that time or is this a a new injury\n[patient] it's new\n[doctor] when did it start\n[patient] probably year and a half ago\n[doctor] okay on both elbows about a year and a half ago\n[patient] yeah\n[doctor] okay have you taken anything for the pain\n[patient] ibuprofen eight hundred milligrams three times a day\n[doctor] okay and does anything make it better or worse\n[patient] the more i

In [24]:
def format_transcript(record):
    dialogue = record["dialogue"].replace("\n", " ")
    note = record["note"].replace("\n", " ")
    transcript = f"Dialogue: {dialogue}\n\nMedical Note: {note}"
    return transcript


@weave.op()
def process_medical_record(record: Dict) -> Dict:
    transcript = format_transcript(record)
    prompt = medical_task.format(transcript=transcript)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": medical_system_prompt},
            {"role": "user", "content": prompt},
        ],
    )

    extracted_info = response.choices[0].message.content

    return {
        "input": transcript,
        "output": extracted_info,
    }


@weave.op()
def generate_medical_data(num_samples: int = N_SAMPLES) -> List[Dict]:
    data = load_medical_data(medical_dataset_url, num_samples)
    processed_data = []

    for record in data:
        processed_record = process_medical_record(record)
        processed_data.append(processed_record)

    return processed_data

In [25]:
results = generate_medical_data()

(67, 4)
🍩 https://wandb.ai/thesisdfki-dfki/eval-course/r/call/01a0cb5f-a4cb-7751-801e-42378233e4f2


OpCallError: Error calling Instructor.create: missing a required argument: 'response_model'

In [ ]:
results[0:2]

In [ ]:
weave.publish(results, name="medical_data_raw")

In [ ]:
client = instructor.patch(openai.OpenAI())

In [ ]:
class MainCriteria(BaseModel):
    word_count: Literal[0, 1] = Field(
        description="1 if the word count is within the limit of 150 words, 0 otherwise",
    )
    presence_of_keys: Literal[0, 1] = Field(
        description="1 if all the six targeted keys (Chief complaint, History of present illness, Physical examination, Symptoms, New medications with dosages, Follow-up instructions) are present, 0 otherwise",
    )
    absence_of_PII: Literal[0, 1] = Field(
        description="1 if no PII is present, 0 otherwise",
    )

In [ ]:
# TODO: Make each desired field a separate annotation


class AnnotationResult(BaseModel):
    annotation: Literal[0, 1] = Field(
        description="Binary score: 1 if the extraction meets all criteria, 0 if it fails on any",
    )
    criteria_annotations: MainCriteria = Field(
        description="A score for each of the main criteria",
    )
    note: str = Field(
        description="Brief explanation of the annotation decision, highlighting any issues or exemplary aspects",
    )


annotation_prompt = """
    Review the following medical data extraction task results:

    Task System Prompt:
    {medical_system_prompt}

    Task:
    {medical_task}

    Input:
    {input_text}

    Output:
    {output_text}

    Evaluate the extraction based on these criteria. Only refer to the Output in your evaluation and NOT the Medical Note field:
    1. Completeness: All required fields addressed (Chief complaint, History of present illness, Physical examination, Symptoms, New medications with dosages, Follow-up instructions)
    2. Accuracy: Information correctly extracted from input
    3. Format: Proper bullet list format used (•key: value)
    4. Privacy: No personal identifiable information (PII) included
    5. Conciseness: ~150 words, key information summarized
    6. Use of "N/A" for missing information

    Provide:
    1. Annotation: 1 if the extraction meets all criteria, 0 if it fails on any
    2. Note: Brief explanation of your decision, highlighting any issues or exemplary aspects
"""

annotation_system_prompt = """
You are an AI assistant tasked with evaluating medical data extraction results.
"""

In [ ]:
@weave.op()
def process_annotation(input_text: str, output_text: str) -> AnnotationResult:
    prompt = annotation_prompt.format(
        medical_system_prompt=medical_system_prompt,
        medical_task=medical_task,
        input_text=input_text,
        output_text=output_text,
    )

    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": annotation_system_prompt},
            {"role": "user", "content": prompt},
        ],
        response_model=AnnotationResult,
    )

In [ ]:
DataPoint = Tuple[
    dict,
    dict,
    Literal[0, 1],
    MainCriteria,
    str,
    Optional[str],
    Optional[str],
]


@weave.op()
def generate_annotations(results: List[Dict]) -> List[DataPoint]:
    annotations = []

    for result in results:
        input_text = result["input"]
        output_text = result["output"]
        annotation_result = process_annotation(input_text, output_text)

        combined_task_description = (
            f"System Prompt: {medical_system_prompt}\n\nTask: {medical_task}"
        )

        data_point: DataPoint = (
            {"input": input_text},  # input
            {"output": output_text},  # output
            annotation_result.annotation,  # annotation (1 for correct, 0 for incorrect)
            annotation_result.criteria_annotations.model_dump(),  # criteria_annotations
            annotation_result.note,  # note
            combined_task_description,  # human_description_for_task_or_judge
            "word count, presence of the six targeted keys, and absence of PII, with the first two implemented via code- based assertions and the last via an LLM evaluator",  # human_description_for_metric_details
        )

        annotations.append(data_point)

    return annotations

In [ ]:
annotations = generate_annotations(results)

In [ ]:
annotations[0]

In [ ]:
weave.publish(annotations, name="medical_data_annotations")